In [2]:
import cv2
import os
import numpy as np

def process_videos(root_path, threshold=None, mode="train"):
    folders = sorted([
        f for f in os.listdir(root_path)
        if os.path.isdir(os.path.join(root_path, f)) and not f.endswith("_gt")
    ])

    all_scores = []

    for folder in folders:
        print(f"\nProcessing {mode.upper()} video: {folder}")
        folder_path = os.path.join(root_path, folder)

        frame_files = sorted([
            f for f in os.listdir(folder_path)
            if f.endswith(".tif")
        ])

        if len(frame_files) == 0:
            print(f"Skipping {folder} (no frames found)")
            continue

        prev_frame = cv2.imread(os.path.join(folder_path, frame_files[0]))
        prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)

        video_scores = []

        for i in range(1, len(frame_files)):
            frame = cv2.imread(os.path.join(folder_path, frame_files[i]))
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

            flow = cv2.calcOpticalFlowFarneback(
                prev_gray, gray,
                None,
                0.5, 3, 15, 3, 5, 1.2, 0
            )

            magnitude, _ = cv2.cartToPolar(flow[..., 0], flow[..., 1])
            motion_score = np.mean(magnitude)
            video_scores.append(motion_score)

            # 🔥 TEST MODE → anomaly check
            if mode == "test" and threshold is not None:
                if motion_score > threshold:
                    label = "ANOMALY"
                    color = (0, 0, 255)
                else:
                    label = "Normal"
                    color = (0, 255, 0)

                cv2.putText(
                    frame,
                    f"{label} | Score: {motion_score:.3f}",
                    (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.8,
                    color,
                    2
                )

            cv2.imshow(f"{mode.upper()} Video", frame)
            prev_gray = gray

            if cv2.waitKey(40) & 0xFF == ord('q'):
                cv2.destroyAllWindows()
                exit()

        all_scores.append(video_scores)
        cv2.waitKey(500)

    cv2.destroyAllWindows()
    return all_scores


In [3]:
train_root = r"UCSD_Anomaly_Dataset.v1p2/UCSDped1/Train"

train_scores = process_videos(train_root, mode="train")

flat_train_scores = np.concatenate(train_scores)
mean_motion = flat_train_scores.mean()
std_motion = flat_train_scores.std()
THRESHOLD = mean_motion + 3 * std_motion

print(f"Final Threshold: {THRESHOLD:.4f}")



Processing TRAIN video: Train001

Processing TRAIN video: Train002

Processing TRAIN video: Train003

Processing TRAIN video: Train004

Processing TRAIN video: Train005

Processing TRAIN video: Train006

Processing TRAIN video: Train007

Processing TRAIN video: Train008

Processing TRAIN video: Train009

Processing TRAIN video: Train010

Processing TRAIN video: Train011

Processing TRAIN video: Train012

Processing TRAIN video: Train013

Processing TRAIN video: Train014

Processing TRAIN video: Train015

Processing TRAIN video: Train016

Processing TRAIN video: Train017

Processing TRAIN video: Train018

Processing TRAIN video: Train019

Processing TRAIN video: Train020

Processing TRAIN video: Train021

Processing TRAIN video: Train022

Processing TRAIN video: Train023

Processing TRAIN video: Train024

Processing TRAIN video: Train025

Processing TRAIN video: Train026

Processing TRAIN video: Train027

Processing TRAIN video: Train028

Processing TRAIN video: Train029

Processing TR

In [4]:
test_root = r"UCSD_Anomaly_Dataset.v1p2/UCSDped1/Test"

test_scores = process_videos(
    test_root,
    threshold=THRESHOLD,
    mode="test"
)



Processing TEST video: Test001

Processing TEST video: Test002

Processing TEST video: Test003

Processing TEST video: Test004

Processing TEST video: Test005

Processing TEST video: Test006

Processing TEST video: Test007

Processing TEST video: Test008

Processing TEST video: Test009

Processing TEST video: Test010

Processing TEST video: Test011

Processing TEST video: Test012

Processing TEST video: Test013

Processing TEST video: Test014

Processing TEST video: Test015

Processing TEST video: Test016

Processing TEST video: Test017

Processing TEST video: Test018

Processing TEST video: Test019

Processing TEST video: Test020

Processing TEST video: Test021

Processing TEST video: Test022

Processing TEST video: Test023

Processing TEST video: Test024

Processing TEST video: Test025

Processing TEST video: Test026

Processing TEST video: Test027

Processing TEST video: Test028

Processing TEST video: Test029

Processing TEST video: Test030

Processing TEST video: Test031

Process

In [5]:
import cv2
import numpy as np
import os

def has_gt_anomaly(gt_frame_path):
    """
    Returns True if GT mask contains anomaly
    """
    gt_img = cv2.imread(gt_frame_path, cv2.IMREAD_GRAYSCALE)
    if gt_img is None:
        return False
    return np.any(gt_img > 0)


In [7]:
test_root = r"UCSD_Anomaly_Dataset.v1p2/UCSDped1/Test"

TP = FP = FN = TN = 0
evaluated_frames = 0

test_folders = sorted([
    f for f in os.listdir(test_root)
    if os.path.isdir(os.path.join(test_root, f)) and not f.endswith("_gt")
])

for folder in test_folders:
    video_path = os.path.join(test_root, folder)
    gt_path = os.path.join(test_root, folder + "_gt")

    # 🚨 Skip if GT does not exist
    if not os.path.exists(gt_path):
        print(f"Skipping {folder} (no ground truth available)")
        continue

    print(f"\nEvaluating {folder}")

    frame_files = sorted([f for f in os.listdir(video_path) if f.endswith(".tif")])
    gt_files = sorted([
        f for f in os.listdir(gt_path)
        if f.endswith(".bmp") or f.endswith(".tif")
    ])

    prev_frame = cv2.imread(os.path.join(video_path, frame_files[0]))
    prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)

    for i in range(1, min(len(frame_files), len(gt_files) + 1)):
        frame = cv2.imread(os.path.join(video_path, frame_files[i]))
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        flow = cv2.calcOpticalFlowFarneback(
            prev_gray, gray,
            None,
            0.5, 3, 15, 3, 5, 1.2, 0
        )

        magnitude, _ = cv2.cartToPolar(flow[..., 0], flow[..., 1])
        motion_score = np.mean(magnitude)

        pred_anomaly = motion_score > THRESHOLD
        gt_anomaly = has_gt_anomaly(os.path.join(gt_path, gt_files[i - 1]))

        if pred_anomaly and gt_anomaly:
            TP += 1
        elif pred_anomaly and not gt_anomaly:
            FP += 1
        elif not pred_anomaly and gt_anomaly:
            FN += 1
        else:
            TN += 1

        evaluated_frames += 1
        prev_gray = gray


Skipping Test001 (no ground truth available)
Skipping Test002 (no ground truth available)

Evaluating Test003

Evaluating Test004
Skipping Test005 (no ground truth available)
Skipping Test006 (no ground truth available)
Skipping Test007 (no ground truth available)
Skipping Test008 (no ground truth available)
Skipping Test009 (no ground truth available)
Skipping Test010 (no ground truth available)
Skipping Test011 (no ground truth available)
Skipping Test012 (no ground truth available)
Skipping Test013 (no ground truth available)

Evaluating Test014
Skipping Test015 (no ground truth available)
Skipping Test016 (no ground truth available)
Skipping Test017 (no ground truth available)

Evaluating Test018

Evaluating Test019
Skipping Test020 (no ground truth available)

Evaluating Test021

Evaluating Test022

Evaluating Test023

Evaluating Test024
Skipping Test025 (no ground truth available)
Skipping Test026 (no ground truth available)
Skipping Test027 (no ground truth available)
Skipping T

In [8]:
precision = TP / (TP + FP + 1e-6)
recall = TP / (TP + FN + 1e-6)
f1 = 2 * precision * recall / (precision + recall + 1e-6)

print("Evaluation Results")
print("------------------")
print(f"TP: {TP}")
print(f"FP: {FP}")
print(f"FN: {FN}")
print(f"TN: {TN}")
print(f"Precision: {precision:.3f}")
print(f"Recall   : {recall:.3f}")
print(f"F1-score : {f1:.3f}")



Evaluation Results
------------------
TP: 72
FP: 4
FN: 1160
TN: 754
Precision: 0.947
Recall   : 0.058
F1-score : 0.110


In [9]:
import json

model_data = {
    "model_type": "optical_flow_statistical",
    "mean_motion": float(mean_motion),
    "std_motion": float(std_motion),
    "threshold": float(THRESHOLD),
    "k_value": 3,
    "dataset": "UCSDped1",
    "description": "Baseline crowd anomaly detection model"
}

with open("trained_model.json", "w") as f:
    json.dump(model_data, f, indent=4)

print("✅ Trained model saved as trained_model.json")


✅ Trained model saved as trained_model.json
